# Gemma 4 31B Claim-Level SFT

This Kaggle notebook fine-tunes Gemma 4 31B Instruct on the exported claim-level chat dataset produced by the local `gemma4/` workflow.


In [ ]:
!pip install -q unsloth transformers datasets trl peft accelerate

In [ ]:
import os
os.environ['UNSLOTH_STABLE_DOWNLOADS'] = '1'

from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback
from unsloth import FastLanguageModel, is_bfloat16_supported

MODEL_NAME = 'unsloth/gemma-4-31B-it-unsloth-bnb-4bit'
TRAIN_PATH = '/kaggle/input/nlp-dataset/gemma4/data/exports/claim_sft_train_chat.jsonl'
VAL_PATH = '/kaggle/input/nlp-dataset/gemma4/data/exports/claim_sft_val_chat.jsonl'
OUTPUT_DIR = '/kaggle/working/gemma4_claim_sft_lora'
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 64,
    lora_dropout = 0.05,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
    max_seq_length = MAX_SEQ_LENGTH,
)


In [ ]:
train_dataset = load_dataset('json', data_files=TRAIN_PATH, split='train')
val_dataset = load_dataset('json', data_files=VAL_PATH, split='train')

def format_batch(examples):
    texts = []
    for messages in examples['messages']:
        texts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize = False,
                add_generation_prompt = False,
            )
        )
    return {'text': texts}

train_dataset = train_dataset.map(format_batch, batched=True)
val_dataset = val_dataset.map(format_batch, batched=True)


In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        dataset_text_field = 'text',
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.05,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        weight_decay = 0.01,
        optim = 'adamw_8bit',
        lr_scheduler_type = 'cosine',
        logging_steps = 10,
        save_strategy = 'epoch',
        eval_strategy = 'epoch',
        load_best_model_at_end = True,
        metric_for_best_model = 'eval_loss',
        greater_is_better = False,
        output_dir = OUTPUT_DIR,
        seed = 42,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        packing = False,
    ),
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 1)],
)

try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part = '<start_of_turn>user\n',
        response_part = '<start_of_turn>model\n',
    )
    print('Enabled assistant-only loss masking.')
except Exception as exc:
    print('Could not enable assistant-only loss automatically:', exc)
    print('Continuing with standard conversational loss.')

trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
